<!-- nav -->
[← 7 · Generating test data](07_Generating_Test_Data.ipynb) · [Home](../../README.md) · [Equivalence →](../../docs/EQUIVALENCE.md)

# 8 · Rewriting step by step

**The problem.** Generating code from a model, migrating data, tidying a catalogue: each is a sequence of rewrites,
and many of them decide something on the way, which case to write a label in, what size to give an item. A script
decides silently, and next time perhaps differently. With people and agents in the loop, each decision must be
visible, made by whoever should make it, and recorded, so that a rerun makes it the same way.

A **transform** is a rewrite with a *before* and an *after*, both predicates over the same symbols, and parameters for
what it decides. A **session** applies transforms to a store one decision at a time, and records every step as data.
This case study labels and sizes the items on a shelf; its small schemas are defined here, in the first cell.

In [1]:
import { Expressions as E } from "@mbse/expressions";
import { Predicates, Queries, Transforms } from "@mbse/patterns";
import { JSON as SchemaJSON, Proxies, Schemas, Stores } from "@mbse/schemas/Framework";

const Held = new Schemas.OfRelation.Builder().name("Held").links("shelf", "item").create();
const Shelf = new Schemas.OfObject.Builder().name("Shelf").ref().singleton("Shelf")
  .relations((r) => r.name("items").of(Held).me("shelf")).create();
const Item = new Schemas.OfObject.Builder().name("Item").ref()
  .properties((p) => p.name("name").of((t) => t.as_native(String)),
    (p) => p.name("label").of((t) => t.as_native(String)),
    (p) => p.name("size").of((t) => t.as_native(BigInt)))
  .relations((r) => r.name("shelves").of(Held).me("item")).create();

/** A store whose shelf holds an item of each name, in order: its data is what the shelf reaches. */
function shelf(...names: string[]): any {
  const store: any = new Proxies.OfStore();
  for (const schema of [Shelf, Item, Held]) store.register(schema);
  for (const name of names) {
    const item = store.Item().name(name).create();
    store.Shelf(store.singleton("Shelf")).items((e: any) => e.item(item)).update();
  }
  return store;
}

const i = E.variable("i");
const Named = new Predicates.OfPredicate.Builder().name("Named").symbols({ i: Item }).requires(i.has("name")).create();
const Labelled = new Predicates.OfPredicate.Builder().name("Labelled").symbols({ i: Item }).requires(i.has("label")).create();
const Sized = new Predicates.OfPredicate.Builder().name("Sized").symbols({ i: Item }).requires(i.has("size")).create();

<a id="T08.1"></a>
## 08.1 · a transform

A transform has a name, a before and an after, the **parameters** it decides, and a **rewrite** that makes its after
hold. `Label` writes an item's name as its label, in upper or lower case: `case` is a parameter whose schema is a union
of two options, so it has two values. `Size` gives a labelled item a size, an int, which has too many values to list:

In [2]:
const Option = new Schemas.OfObject.Builder().create();
const Case = new Schemas.OfUnion.Builder().branches((b) => b.name("upper").of(Option), (b) => b.name("lower").of(Option)).create();

function label(store: any, match: Transforms.Match, args: Record<string, unknown>): void {
  const item = match["i"] as any;
  store.Item(item).label(args["case"] === "upper" ? item.name.toUpperCase() : item.name.toLowerCase()).update();
}

function size(store: any, match: Transforms.Match, args: Record<string, unknown>): void {
  store.Item(match["i"]).size(args["size"]).update();
}

const Label = new Transforms.Transform("Label", Named, Labelled, { parameters: [(p) => p.name("case").of(Case)], rewrite: label });
const Size = new Transforms.Transform("Size", Labelled, Sized,
  { parameters: [(p) => p.name("size").of((t) => t.as_native(BigInt))], rewrite: size });
console.log(Label.check(), Size.check(), [...Label.parameters.keys()], [...Size.parameters.keys()]);

[] [] [ "case" ] [ "size" ]


`check()` finds nothing wrong: the before and after have the same symbols, and each transform has a rewrite.

<a id="T08.2"></a>
## 08.2 · candidates

A session offers **candidates**: a transform at a match where its before holds and its after doesn't. Each value of a
finite parameter is a candidate of its own, so bolt and nut each have two; `Size` has none yet, since nothing is
labelled:

In [3]:
let store = shelf("bolt", "nut");
let session = new Transforms.Session(store, [Label, Size]);
const seen = (candidates: Transforms.Candidate[]) => candidates.map((c) => [c.transform.name, (c.match["i"] as any).name, c.arguments]);
console.log(seen(session.candidates()));

[
  [ "Label", "bolt", { case: "upper" } ],
  [ "Label", "bolt", { case: "lower" } ],
  [ "Label", "nut", { case: "upper" } ],
  [ "Label", "nut", { case: "lower" } ]
]


A candidate decides everything but its **open** parameters, those with no finite domain, which whoever takes it must
answer.

<a id="T08.3"></a>
## 08.3 · taking a step yourself

The caller decides: `take` applies one candidate, one decision. The data changes, and so do the candidates: bolt's
other case is gone, and its size is enabled, open until you answer it:

In [4]:
session.take(session.candidates()[0] as Transforms.Candidate); // bolt, upper
const sizing = session.candidates()[2] as Transforms.Candidate;
console.log(seen(session.candidates()), sizing.open);
session.take(sizing.answer({ size: 3n }));
const items = (store: any) => store.extent("Item").map((item: any) => [item.name, item.label ?? null, item.size ?? null])
  .sort((a: string[], b: string[]) => (a[0]! < b[0]! ? -1 : 1));
console.log(items(store));

[
  [ "Label", "nut", { case: "upper" } ],
  [ "Label", "nut", { case: "lower" } ],
  [ "Size", "bolt", {} ]
] [ "size" ]


[ [ "bolt", "BOLT", 3n ], [ "nut", null, null ] ]


<a id="T08.4"></a>
## 08.4 · a rewrite must establish its after

A session checks each step: after the rewrite, the after must hold. A rewrite that doesn't do what its transform says
is refused, by name and match:

In [5]:
const Forgetful = new Transforms.Transform("Forgetful", Named, Labelled, { rewrite: () => null });
const forgetting = new Transforms.Session(shelf("bolt"), [Forgetful]);
try {
  forgetting.take(forgetting.candidates()[0] as Transforms.Candidate);
} catch (error) {
  console.log(String(error));
}

ValueError: 'Forgetful' did not establish its after at i=Shelf/items[0]


<a id="T08.5"></a>
## 08.5 · a policy decides

Where you'd rather not decide each step, a **policy** does. Its clauses weigh candidates by transform and arguments,
and answer open parameters; `candidates(policy)` ranks them, a recommendation:

In [6]:
const policy = new Transforms.Policy(new Transforms.Clause("Label", { case: "lower" }, 2), new Transforms.Clause("Size", { size: 5n }));
store = shelf("bolt", "nut");
session = new Transforms.Session(store, [Label, Size]);
console.log(session.candidates(policy).map((c) => [c.transform.name, (c.match["i"] as any).name, c.arguments, c.score]));

[
  [ "Label", "bolt", { case: "lower" }, 2 ],
  [ "Label", "nut", { case: "lower" }, 2 ],
  [ "Label", "bolt", { case: "upper" }, 0 ],
  [ "Label", "nut", { case: "upper" }, 0 ]
]


`step_over(policy)` takes the policy's first candidate, and `run(policy)` steps until the policy has nothing it can
take; a candidate no clause weighs is never the policy's to take:

In [7]:
console.log(session.step_over(policy), session.run(policy), session.done, session.steps.map((step) => step.by));
console.log(items(store));

true 3 true [ "policy", "policy", "policy", "policy" ]


[ [ "bolt", "bolt", 5n ], [ "nut", "nut", 5n ] ]


<a id="T08.6"></a>
## 08.6 · composite transforms, stepped into or over

A transform with **parts** instead of a rewrite is composite: `Finish` is `Label`, then `Size`. Step into it, and you
decide inside it, for its match alone; step over it, and the policy does:

In [8]:
const Finished = new Predicates.OfPredicate.Builder().name("Finished").symbols({ i: Item })
  .requires(i.has("label").and_(i.has("size"))).create();
const Finish = new Transforms.Transform("Finish", Named, Finished, { parts: [Label, Size] });
session = new Transforms.Session(shelf("bolt", "nut"), [Finish]);
const inner = session.step_in(session.candidates()[0] as Transforms.Candidate); // bolt: the caller decides inside
inner.take(inner.candidates()[1] as Transforms.Candidate); // lower
inner.take((inner.candidates()[0] as Transforms.Candidate).answer({ size: 2n }));
session.step_over(new Transforms.Policy(new Transforms.Clause("Finish"), new Transforms.Clause("Label", { case: "upper" }),
  new Transforms.Clause("Size", { size: 4n }))); // nut: the policy decides
console.log(session.steps.map((s) => [s.transform, s.by, s.mode, s.steps.map((x) => [x.transform, x.by, Object.fromEntries(x.arguments)])]));

[
  [
    "Finish",
    "caller",
    "in",
    [
      [ "Label", "caller", { case: "lower" } ],
      [ "Size", "caller", { size: 2n } ]
    ]
  ],
  [
    "Finish",
    "policy",
    "over",
    [
      [ "Label", "policy", { case: "upper" } ],
      [ "Size", "policy", { size: 4n } ]
    ]
  ]
]


Each step records who decided it, `caller` or `policy`, and a composite's steps hold its parts' steps.

<a id="T08.7"></a>
## 08.7 · the trace is data

The steps taken are a **trace**, written as data like any other, and read back as steps. Each step's **key** names its
transform and where it matched, by path from the store's roots:

In [9]:
const traces = Transforms.register(new Proxies.OfStore()); // a store of the trace's schemas
const text = SchemaJSON.ToJSON(traces)(Transforms.Trace, session.trace(traces) as never);
console.log(text.slice(0, 300), "...");
console.log(Transforms.steps(traces, session.trace(traces)).map((step) => step.key));

{"root": "s0", "objects": {"s0": {"steps": [{"transform": "Finish", "match": [{"symbol": "i", "element": "Shelf/items[0]"}], "by": "caller", "mode": "in", "steps": [{"transform": "Label", "match": [{"symbol": "i", "element": "Shelf/items[0]"}], "arguments": [{"name": "case", "value": {"str": "lower" ...


[ "Finish(i=Shelf/items[0])", "Finish(i=Shelf/items[1])" ]


<a id="T08.8"></a>
## 08.8 · a rerun reuses decisions

A rerun given the earlier steps takes each earlier decision again where its key still occurs, here on a store with
one more item. `run()` with no policy takes only those; the new item's steps are still to decide:

In [10]:
const decide = new Transforms.Policy(new Transforms.Clause("Label", { case: "upper" }), new Transforms.Clause("Size", { size: 2n }));
const first = new Transforms.Session(shelf("a", "b"), [Label, Size]);
first.run(decide);
const again = new Transforms.Session(shelf("a", "b", "c"), [Label, Size], {}, null, first.steps);
console.log(again.run(), again.steps.map((step) => step.by), again.done,
  again.candidates().slice(0, 1).map((c) => [c.transform.name, (c.match["i"] as any).name]));

4 [ "reused", "reused", "reused", "reused" ] false [ [ "Label", "c" ] ]


A decision whose key no longer occurs is an **orphan**, reported rather than dropped, and `diff` compares two runs by
key: the steps only the later took (here, the new item's), those only the earlier took, and those both took with other
arguments (here, a run that chose lower case):

In [11]:
again.run(new Transforms.Policy(new Transforms.Clause("Label", { case: "lower" }), new Transforms.Clause("Size", { size: 3n })));
const fewer = new Transforms.Session(shelf("a"), [Label, Size], {}, null, first.steps);
fewer.run();
const lower = new Transforms.Session(shelf("a", "b"), [Label, Size]);
lower.run(new Transforms.Policy(new Transforms.Clause("Label", { case: "lower" }), new Transforms.Clause("Size", { size: 2n })));
console.log(fewer.orphans.map((s) => s.key));
console.log(Transforms.diff(first.steps, again.steps).added.map((s) => s.key));
console.log(Transforms.diff(first.steps, lower.steps).changed.map(([a, b]) => [a.key, Object.fromEntries(a.arguments), Object.fromEntries(b.arguments)]));

[ "Label(i=Shelf/items[1])", "Size(i=Shelf/items[1])" ]


[ "Label(i=Shelf/items[2])", "Size(i=Shelf/items[2])" ]


[
  [ "Label(i=Shelf/items[0])", { case: "upper" }, { case: "lower" } ],
  [ "Label(i=Shelf/items[1])", { case: "upper" }, { case: "lower" } ]
]


<a id="T08.9"></a>
## 08.9 · what each step wrote

A rewrite may say what it wrote, by role; the step links what it matched and what it wrote, so you can ask which step
made something. `Pack` boxes each item: a new box on the shelf, labelled with the item's name:

In [12]:
/** Puts a box for the item on the shelf, labels the item packed, and says what it wrote, by role. */
function pack(store: any, match: Transforms.Match): Map<string, unknown> {
  const item = match["i"] as any;
  const box = store.Item().name("box").label(item.name).create();
  store.Shelf(store.singleton("Shelf")).items((e: any) => e.item(box)).update();
  store.Item(item).label("packed").update();
  return new Map<string, unknown>([["box", box], ["item", item]]);
}

const Unlabelled = new Predicates.OfPredicate.Builder().name("Unlabelled").symbols({ i: Item })
  .requires(i.has("name")).forbids(i.has("label")).create();
const Pack = new Transforms.Transform("Pack", Unlabelled, Labelled, { rewrite: pack });
const packing = new Transforms.Session(shelf("bolt", "nut"), [Pack]);
packing.run(new Transforms.Policy(new Transforms.Clause("Pack")));
const bolts = packing.steps[0] as Transforms.Step;
const box = (bolts.wrote[0] as Transforms.Link).element as any;
console.log(bolts.wrote.map((link) => link.role), box.label, packing.wrote(box) === bolts);

[ "box", "item" ] bolt true


In a trace, each link is a path. Steps are objects too: combined with the session's store, a predicate queries them
like any other data, here each box and the step that made it (`Stores.Combined` is mbse-schemas 0.10.0's, which this
release pins):

In [13]:
console.log((Transforms.steps(traces, packing.trace(traces))[0] as Transforms.Step).wrote.map((link) => [link.role, link.path]));
const combined = new Stores.Combined(packing.store, Transforms.records(packing));
const [st, b] = [E.variable("st"), E.variable("b")];
const Boxed = new Predicates.OfPredicate.Builder().name("Boxed").symbols({ st: Transforms.StepObject, b: Item })
  .requires(Predicates.Contains(st.get("wrote"), (e) => e.element.eq(b).and_(e.role.eq("box")))).create();
console.log([...Queries.select(combined, Boxed)].map((m: any) => [m["st"].key, m["b"].label]));

[ [ "box", "Shelf/items[0]" ], [ "item", "Shelf/items[1]" ] ]


[
  [ "Pack(i=Shelf/items[0])", "bolt" ],
  [ "Pack(i=Shelf/items[1])", "nut" ]
]


## Why it works this way

- **Every decision is a step.** A rewrite decides only what its parameters say, so each decision is visible, made by the
  caller or delegated to a policy, and recorded: [CAP-074: transforms applied step by step](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-074.md); [PRI-014: every ambiguity a decision](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-014).
- **Before and after are predicates.** A transform is enabled where its before holds and its after doesn't, so a step
  that is done is never offered again, and a rewrite that fails its after is refused.
- **Links, not copies.** A step links what it matched and wrote, so what a step made can be traced back to it:
  [CAP-075: steps linked to what they matched and wrote](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-075.md).
- **Keys make reruns cheap.** A step's key is its transform and its match's paths, so a rerun on changed data takes
  every decision that still applies and reports the rest: [CAP-076: reruns reuse decisions](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-076.md).

<!-- nav -->
[← 7 · Generating test data](07_Generating_Test_Data.ipynb) · [Home](../../README.md) · [Equivalence →](../../docs/EQUIVALENCE.md)